# Week 2 — SNLI-VE Visual Entailment (N=1000)
**Project**: Quantum Kernels on CLIP/BERT Embeddings — ICLR Extension

**Dataset**: SNLI-VE — Visual Entailment
Given an image (premise) + text hypothesis, classify the relationship as:
entailment (0) / neutral (1) / contradiction (2)

**Why this dataset is unique in the paper:**
This is the only task using JOINT multimodal embeddings. CLIP encodes both the
image (512-dim) and the hypothesis text (512-dim), which are concatenated into
a 1024-dim joint embedding before PCA reduction. Neither pure-text BERT nor
pure-image CLIP embeddings capture this cross-modal interaction.

**Metric:** macro F1 (3-class, balanced)
**Runtime:** ~15 min (CLIP extracts both image AND text features per sample)

In [ ]:
!pip install datasets transformers torch Pillow scikit-learn numpy --quiet
import numpy as np, time, os, pickle, shutil, json
import matplotlib.pyplot as plt
import warnings; warnings.filterwarnings('ignore')
import torch
from transformers import CLIPModel, CLIPProcessor
from sklearn.decomposition import PCA
from sklearn.svm import SVC
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import accuracy_score, f1_score
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Device: {DEVICE}")

---
## Step 0: Storage + helpers

In [ ]:
USE_DRIVE = True
if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    SAVE_DIR = '/content/drive/MyDrive/qml_project'
else:
    SAVE_DIR = 'qml_project'
os.makedirs(SAVE_DIR, exist_ok=True)
TAG = 'snlive'
print(f"Save dir: {SAVE_DIR} | Tag: {TAG}")

def _persist_and_download(path):
    if os.path.abspath(SAVE_DIR) != os.getcwd():
        shutil.copy(path, os.path.join(SAVE_DIR, os.path.basename(path)))
        print(f"  saved -> {SAVE_DIR}/{os.path.basename(path)}")
    try:
        from google.colab import files
        files.download(path)
        print(f"  download -> {os.path.basename(path)}")
    except Exception as e:
        print(f"  (download skipped: {type(e).__name__} — saved above)")
def checkpoint(filename, save_fn): save_fn(filename); _persist_and_download(filename)
def save_figure(fig, name):
    for ext in ('png','pdf'):
        p = f"{name}.{ext}"
        fig.savefig(p, bbox_inches='tight', dpi=150); _persist_and_download(p)
def _find(fname):
    dp = os.path.join(SAVE_DIR, fname)
    return dp if os.path.exists(dp) else fname
print("Helpers ready.")

---
## Step 1: Load SNLI-VE

Primary path: `HuggingFaceM4/SNLI-VE` — includes Flickr30k images.
We load 1100 train + 250 test, balanced across all 3 classes.
Samples with label=-1 (no consensus) are filtered out.

In [ ]:
from datasets import load_dataset

print("Loading HuggingFaceM4/SNLI-VE (this may take a few minutes — large dataset)...")
dataset = load_dataset("HuggingFaceM4/SNLI-VE")

# Diagnostic: print column names so we know the structure
print(f"\nColumn names: {dataset['train'].column_names}")
print(f"First sample keys: {list(dataset['train'][0].keys())}")

# Detect column names (handle both 'image'/'premise' and 'hypothesis'/'label' variants)
cols = dataset['train'].column_names
img_col  = 'image'       if 'image'      in cols else 'premise'
txt_col  = 'hypothesis'  if 'hypothesis' in cols else 'caption'
lbl_col  = 'label'

print(f"\nUsing columns: image='{img_col}', text='{txt_col}', label='{lbl_col}'")
LABELS = {0: 'entailment', 1: 'neutral', 2: 'contradiction'}

In [ ]:
N_PER_CLASS   = 400   # 400 per class → 1200 total buffer (we'll subsample to 1000/200)
N_TEST_CLASS  = 90    # 90 per class → 270 test buffer (subsample to 200)

def load_balanced(split, n_per_class, img_col, txt_col, lbl_col):
    imgs, txts, lbls = [], [], []
    counts = {0:0, 1:0, 2:0}
    for sample in dataset[split]:
        lbl = sample[lbl_col]
        if lbl not in [0,1,2]: continue       # skip label=-1 (no consensus)
        if counts[lbl] >= n_per_class: continue
        imgs.append(sample[img_col])
        txts.append(sample[txt_col])
        lbls.append(lbl)
        counts[lbl] += 1
        if all(v >= n_per_class for v in counts.values()): break
    return imgs, txts, np.array(lbls)

print("Collecting balanced train samples...")
train_imgs, train_txts, train_labels = load_balanced('train', N_PER_CLASS, img_col, txt_col, lbl_col)
print(f"  Train: {len(train_imgs)} samples ({dict(zip(*np.unique(train_labels, return_counts=True)))})")

print("Collecting balanced test samples...")
test_imgs, test_txts, test_labels = load_balanced('test', N_TEST_CLASS, img_col, txt_col, lbl_col)
print(f"  Test : {len(test_imgs)} samples")

print(f"\nExample:")
print(f"  Hypothesis: {train_txts[0]}")
print(f"  Label: {LABELS[train_labels[0]]}")

---
## Step 2: Load CLIP + extract JOINT embeddings — checkpoint after

This is the unique step. For each sample we extract:
- Image embedding (512-dim) via CLIP vision encoder
- Text embedding (512-dim) via CLIP text encoder
- Concatenate → 1024-dim joint multimodal embedding

This captures cross-modal interaction that neither image-only nor text-only embeddings carry.

In [ ]:
MODEL_NAME = "openai/clip-vit-base-patch32"
clip_processor = CLIPProcessor.from_pretrained(MODEL_NAME)
clip_model     = CLIPModel.from_pretrained(MODEL_NAME).to(DEVICE)
clip_model.eval()
print(f"CLIP loaded. Image dim: 512, Text dim: 512, Joint dim: 1024")

In [ ]:
def extract_clip_joint(images, texts, batch_size=32):
    """Extract joint 1024-dim embeddings: [image_features; text_features]."""
    all_embeddings = []
    nb = (len(images) + batch_size - 1) // batch_size
    for i in range(0, len(images), batch_size):
        img_batch = images[i:i+batch_size]
        txt_batch = texts[i:i+batch_size]

        # Image features — using direct vision_model (avoids BaseModelOutputWithPooling error)
        img_inputs   = clip_processor(images=img_batch, return_tensors='pt')
        pixel_values = img_inputs['pixel_values'].to(DEVICE)
        with torch.no_grad():
            vision_out = clip_model.vision_model(pixel_values=pixel_values)
            img_emb    = clip_model.visual_projection(vision_out.pooler_output)  # (B, 512)
            img_emb    = img_emb / torch.linalg.norm(img_emb, dim=-1, keepdim=True)

        # Text features via CLIP tokenizer
        txt_inputs = clip_processor.tokenizer(
            txt_batch, return_tensors='pt', padding=True, truncation=True, max_length=77
        ).to(DEVICE)
        with torch.no_grad():
            txt_emb = clip_model.get_text_features(**txt_inputs)   # (B, 512)
            txt_emb = txt_emb / torch.linalg.norm(txt_emb, dim=-1, keepdim=True)

        # Concatenate image + text → 1024-dim joint embedding
        joint = torch.cat([img_emb, txt_emb], dim=-1)   # (B, 1024)
        all_embeddings.append(joint.cpu().numpy())

        if (i//batch_size+1) % 5 == 0: print(f"  batch {i//batch_size+1}/{nb}")
    return np.vstack(all_embeddings)

print("Extracting train JOINT embeddings (image + text)...")
t0 = time.time()
train_emb = extract_clip_joint(train_imgs, train_txts)
print(f"  done {time.time()-t0:.0f}s | {train_emb.shape}  (joint 1024-dim)")

print("\nExtracting test JOINT embeddings...")
test_emb = extract_clip_joint(test_imgs, test_txts)
print(f"  done | {test_emb.shape}")

assert train_emb.shape[1] == 1024, "Expected 1024-dim joint embedding"
print(f"\nJoint embedding confirmed: 1024-dim")

print("\nCheckpoint 1 — raw joint embeddings:")
checkpoint(f'snlive_raw.npz',
           lambda p: np.savez(p, train_emb=train_emb, test_emb=test_emb,
                              train_labels=train_labels, test_labels=test_labels))

---
## Step 3: PCA (1024 → 8) + normalize — checkpoint after

In [ ]:
N_QUBITS = 8
pca_dims = [4, 8, 12, 16]
pca_objects = {n: PCA(n_components=n, random_state=42).fit(train_emb) for n in pca_dims}

print("PCA variance explained (1024-dim joint CLIP → reduced):")
for n in pca_dims:
    print(f"  {n:>2} dims -> {np.sum(pca_objects[n].explained_variance_ratio_)*100:.1f}%")

pca_8 = pca_objects[N_QUBITS]
X_train_pca = pca_8.transform(train_emb)
X_test_pca  = pca_8.transform(test_emb)

scaler = MinMaxScaler(feature_range=(0, np.pi))
X_train_scaled = scaler.fit_transform(X_train_pca)
X_test_scaled  = np.clip(scaler.transform(X_test_pca), 0, np.pi)

print(f"\nTrain range: [{X_train_scaled.min():.3f}, {X_train_scaled.max():.3f}]")

print("\nCheckpoint 2 — PCA + scaler:")
checkpoint(f'{TAG}_pca_scaler.pkl',
           lambda p: pickle.dump({'pca': pca_8, 'scaler': scaler, 'n_qubits': N_QUBITS,
                                  'embedding_type': 'clip_joint_1024',
                                  'n_classes': 3}, open(p,'wb')))

---
## Step 4: Classical baselines on real 1000 / 200 split

In [ ]:
N_TRAIN_SVM, N_TEST_SVM = 1000, 200
assert len(X_train_scaled) >= N_TRAIN_SVM
assert len(X_test_scaled)  >= N_TEST_SVM

idx_tr = np.random.RandomState(42).choice(len(X_train_scaled), N_TRAIN_SVM, replace=False)
idx_te = np.random.RandomState(42).choice(len(X_test_scaled),  N_TEST_SVM,  replace=False)
X_tr, y_tr = X_train_scaled[idx_tr], train_labels[idx_tr]
X_te, y_te = X_test_scaled[idx_te],  test_labels[idx_te]

def mf1(yt, yp): return float(f1_score(yt, yp, average='macro'))

print(f"Train {X_tr.shape} | Test {X_te.shape} | metric=macro F1")
print(f"Class balance: {dict(zip(*np.unique(y_tr, return_counts=True)))}")
print()

classifiers = {
    'Linear':     SVC(kernel='linear', C=1.0, random_state=42),
    'RBF':        SVC(kernel='rbf',    C=1.0, gamma='scale', random_state=42),
    'Polynomial': SVC(kernel='poly',   C=1.0, degree=3, coef0=1, random_state=42),
}
results = {}
print(f"{'Kernel':<14} {'Accuracy':>10} {'MacroF1':>10}")
print("-"*38)
for name, clf in classifiers.items():
    clf.fit(X_tr, y_tr)
    yp = clf.predict(X_te)
    results[name] = {'accuracy': float(accuracy_score(y_te, yp)), 'f1': mf1(y_te, yp)}
    print(f"{name:<14} {results[name]['accuracy']:>10.4f} {results[name]['f1']:>10.4f}")

# Figure
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
names = list(results.keys())
for ax, key, lbl in [(axes[0],'accuracy','Accuracy'), (axes[1],'f1','Macro F1')]:
    vals = [results[n][key] for n in names]
    ax.bar(names, vals, color=['steelblue','coral','mediumseagreen'], edgecolor='white')
    ax.set_ylim(0.2, 1.0); ax.set_title(lbl); ax.set_ylabel(lbl)
    for i, v in enumerate(vals): ax.text(i, v+0.005, f'{v:.3f}', ha='center', fontsize=11)
plt.suptitle('SNLI-VE Classical Baselines (joint CLIP 1024-dim, N=1000)')
plt.tight_layout(); plt.show()
save_figure(fig, f'{TAG}_baselines')

print("\nCheckpoint 3 — baselines:")
checkpoint(f'{TAG}_baseline_results.pkl', lambda p: pickle.dump(results, open(p,'wb')))

---
## Step 5: Save final embeddings — shape verified

In [ ]:
def save_final(p):
    np.savez(p, X_train=X_train_scaled, X_test=X_test_scaled,
             y_train=train_labels, y_test=test_labels,
             X_tr_subset=X_tr, X_te_subset=X_te,
             y_tr_subset=y_tr, y_te_subset=y_te)

print("Checkpoint 4 — final embeddings:")
checkpoint(f'{TAG}_embeddings.npz', save_final)

loaded = np.load(f'{TAG}_embeddings.npz')
print(f"\n  X_tr_subset = {loaded['X_tr_subset'].shape}  (must be (1000, 8))")
print(f"  X_te_subset = {loaded['X_te_subset'].shape}  (must be (200, 8))")
assert loaded['X_tr_subset'].shape == (1000, 8)
assert loaded['X_te_subset'].shape == (200, 8)
print("  VERIFIED.")

meta = {'dataset':'snlive','embedding_type':'clip_joint_1024','n_classes':3,
        'metric':'macro','n_train':1000,'n_test':200,'baselines':results}
checkpoint(f'{TAG}_meta.json', lambda p: json.dump(meta, open(p,'w'), indent=2))

---
## Week 2 done — SNLI-VE

### Saved:
- `snlive_raw.npz` — 1024-dim joint CLIP embeddings
- `snlive_pca_scaler.pkl`
- `snlive_baseline_results.pkl`
- `snlive_embeddings.npz` — 1000/200 split ready for Week 3
- `snlive_meta.json`
- `snlive_baselines.png/.pdf`

### What makes this unique vs everything else:
The 1024-dim embedding combines image + text through CLIP's shared space.
Quantum kernels may be more or less sensitive to this higher-dimensional
cross-modal structure. That's the hypothesis your Week 3 result will test.